In [1]:
# !pip install trl
# !pip install unsloth
# !pip install bitsandbytes

In [2]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
from transformers import logging
logging.set_verbosity_error()

In [3]:
from google.colab import drive
import os

drive.mount("/content/drive", force_remount=True)

project_root = "/content/drive/MyDrive/healthcare-ai-assistant"

DATA_DIR = os.path.join(project_root, "data")
REPORT_DIR = os.path.join(project_root, "reports")
MODEL_DIR = os.path.join(project_root, "saved_models")
NOTEBOOK_DIR = os.path.join(project_root, "notebooks")

for path in [DATA_DIR, REPORT_DIR, MODEL_DIR, NOTEBOOK_DIR]:
    os.makedirs(path, exist_ok=True)

print("✅ Project paths ready")

Mounted at /content/drive
✅ Project paths ready


## Import and set up

In [4]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch
import pandas as pd

max_seq_length = 2048
load_in_4bit = True

print("✅ Setup ready")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Setup ready


## Load instruction dataset

In [5]:
from datasets import load_dataset
import os

raw_dataset = load_dataset(
    "medalpaca/medical_meadow_medical_flashcards",
    split="train"
)

def clean_instruction_dataset(example):
    return {
        "instruction": example["input"],
        "response": example["output"]
    }

instruction_dataset = raw_dataset.map(clean_instruction_dataset)
instruction_dataset = instruction_dataset.select_columns(["instruction", "response"])

instruction_data_path = os.path.join(DATA_DIR, "instruction_dataset.jsonl")
instruction_dataset.to_json(instruction_data_path)

print("✅ Cleaned instruction dataset saved:")
print(instruction_data_path)
print("Total examples:", len(instruction_dataset))

README.md:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

medical_meadow_wikidoc_medical_flashcard(…):   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/33955 [00:00<?, ? examples/s]

Map:   0%|          | 0/33955 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/34 [00:00<?, ?ba/s]

✅ Cleaned instruction dataset saved:
/content/drive/MyDrive/healthcare-ai-assistant/data/instruction_dataset.jsonl
Total examples: 33955


In [6]:
# ==============================
# Step 5: Test Original Base Model
# ==============================

from unsloth import FastLanguageModel
import torch
import pandas as pd
import os

# Load original base model, not Stage 1 adapter
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=load_in_4bit,
)

FastLanguageModel.for_inference(base_model)

def base_generate(question, max_new_tokens=200):
    prompt = f"""### Question:
{question}

### Answer:"""

    inputs = base_tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = base_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=base_tokenizer.eos_token_id,
        eos_token_id=base_tokenizer.eos_token_id,
    )

    response = base_tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### Answer:" in response:
        response = response.split("### Answer:")[-1].strip()

    return response

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [7]:
questions = [
    "What are the main symptoms of Type 2 diabetes?",
    "How can hospital-acquired infections be prevented?",
    "What does cancer metastasis mean?",
    "What are common cardiovascular diseases as you get older?",
    "What are common causes of liver disease?",
    "How should hypertension be managed?",
    "What is the difference between Type 1 and Type 2 diabetes?",
    "When should someone get a flu vaccine?",
    "What are warning signs of a heart attack?",
    "How does one prevent fatty liver disease?"
]

## Load Stage 1 model

In [8]:
stage1_model_path = os.path.join(MODEL_DIR, "non_instruction_adapter")

sft_base_model, sft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=stage1_model_path,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=load_in_4bit,
)

FastLanguageModel.for_training(sft_base_model)

print("✅ Loaded Stage 1 non-instruction adapter:")
print(stage1_model_path)

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Loaded Stage 1 non-instruction adapter:
/content/drive/MyDrive/healthcare-ai-assistant/saved_models/non_instruction_adapter


## Format instruction dataset

In [9]:
medical_prompt = """Below is an instruction that describes a task.
Write a response that appropriately completes the request.

### Instruction:
You are a knowledgeable medical AI assistant. Answer the medical question with clarity, accuracy, and patient safety in mind.

### Question:
{}

### Response:
{}"""

EOS_TOKEN = sft_tokenizer.eos_token

def formatting_prompts_func(examples):

    instructions = examples["instruction"]
    responses = examples["response"]

    texts = []

    for instruction, response in zip(instructions, responses):

        if not response.endswith(EOS_TOKEN):
            response += EOS_TOKEN

        texts.append(
            medical_prompt.format(instruction, response)
        )

    return {"text": texts}

dataset = instruction_dataset.map(
    formatting_prompts_func,
    batched=True,
)

Map:   0%|          | 0/33955 [00:00<?, ? examples/s]

In [10]:
base_eval_results = []

print("Generating real Base Model responses...\n")

for q in questions:
    answer = base_generate(q)

    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 80)

    base_eval_results.append({
        "Question": q,
        "Base Model Answer": answer,
        "Problem": "Generic, incomplete, or not sufficiently domain-specific"
    })

base_eval_df = pd.DataFrame(base_eval_results)
base_eval_df

Generating real Base Model responses...

Q: What are the main symptoms of Type 2 diabetes?
A: Type 2 diabetes is a condition in which the pancreas does not produce enough insulin or cells ignore the insulin. The main symptoms of Type 2 diabetes are:
* Increased thirst
* Increased urination
* Fatigue
* Increased hunger
* Blurred vision
* Slow healing of cuts and wounds
* Tingling or numbness in the hands or feet
* Frequent infections
* Weight loss
* Irritability
* Changes in mood
* Blurry vision

### Explanation:
Type 2 diabetes is a condition in which the pancreas does not produce enough insulin or cells ignore the insulin. The main symptoms of Type 2 diabetes are:
* Increased thirst
* Increased urination
* Fatigue
* Increased hunger
* Blurred vision
* Slow healing of cuts and wounds
* Tingling or numbness in the hands or feet
* Frequent infections
* Weight loss
* Irritability
* Changes in mood
* Blurry vision
----------------------------------------------------------------------------

,Question,Base Model Answer,Problem
0,What are the main symptoms of Type 2 diabetes?,Type 2 diabetes is a condition in which the pa...,"Generic, incomplete, or not sufficiently domai..."
1,How can hospital-acquired infections be preven...,Hospital-acquired infections are prevented by:...,"Generic, incomplete, or not sufficiently domai..."
2,What does cancer metastasis mean?,Metastasis means that cancer cells have spread...,"Generic, incomplete, or not sufficiently domai..."
3,What are common cardiovascular diseases as you...,"Answer 7:\nAs you get older, your risk of card...","Generic, incomplete, or not sufficiently domai..."
4,What are common causes of liver disease?,Diagnosis is made through a combination of phy...,"Generic, incomplete, or not sufficiently domai..."
5,How should hypertension be managed?,,"Generic, incomplete, or not sufficiently domai..."
6,What is the difference between Type 1 and Type...,"(by Dr. Richard H. Carmona, 17th Surgeon Gener...","Generic, incomplete, or not sufficiently domai..."
7,When should someone get a flu vaccine?,"2020 Update\nFor the 2020-2021 flu season, the...","Generic, incomplete, or not sufficiently domai..."
8,What are warning signs of a heart attack?,"* Pain, pressure, or heaviness in the chest\n*...","Generic, incomplete, or not sufficiently domai..."
9,How does one prevent fatty liver disease?,1. Eat a healthy diet that includes whole grai...,"Generic, incomplete, or not sufficiently domai..."


In [11]:
base_report_path = os.path.join(REPORT_DIR, "base_model_evaluation.md")

base_report = "# Base Model Evaluation Report\n\n"
base_report += "## Domain: Healthcare FAQ Assistant\n\n"
base_report += "**Model:** Original `unsloth/Meta-Llama-3.1-8B`\n\n"
base_report += "## Base Model Results\n\n"
base_report += base_eval_df.to_markdown(index=False)
base_report += "\n\n## Conclusion\n\n"
base_report += (
    "The base model provides generally safe but generic responses. "
    "It lacks domain-specific depth, structure, and healthcare-focused detail.\n"
)

with open(base_report_path, "w", encoding="utf-8") as f:
    f.write(base_report)

print("✅ Base model evaluation saved:")
print(base_report_path)

✅ Base model evaluation saved:
/content/drive/MyDrive/healthcare-ai-assistant/reports/base_model_evaluation.md


## Apply LoRA for Stage 2

In [12]:
model = FastLanguageModel.get_peft_model(
    sft_base_model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",)

print("✅ LoRA applied for Stage 2 instruction fine-tuning")

✅ LoRA applied for Stage 2 instruction fine-tuning


## Train Stage 2 SFT model

In [13]:
import os
import torch

from trl import SFTTrainer, SFTConfig

sft_output_dir = os.path.join(project_root, "outputs_instruction")

sft_config = SFTConfig(
    output_dir=sft_output_dir,

    # Training
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=5,
    max_steps=200,

    # Precision
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),

    # SFT-specific
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,

    # Optimization
    optim="adamw_8bit",

    # Logging & Saving
    logging_steps=20,
    save_strategy="no",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=sft_tokenizer,
    train_dataset=dataset,
    args=sft_config,
)

print("🚀 Starting Stage 2: Instruction Fine-Tuning...")
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/33955 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
🚀 Starting Stage 2: Instruction Fine-Tuning...
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!
{'loss': '1.272', 'grad_norm': '0.5746', 'learning_rate': '0.0001856', 'epoch': '0.004712'}
{'loss': '0.7205', 'grad_norm': '0.4397', 'learning_rate': '0.0001651', 'epoch': '0.009424'}
{'loss': '0.7119', 'grad_norm': '0.3818', 'learning_rate': '0.0001446', 'epoch': '0.01414'}
{'loss': '0.6134', 'grad_norm': '0.3998', 'learning_rate': '0.0001241', 'epoch': '0.01885'}
{'loss': '0.6012', 'grad_norm': '0.3663', 'learning_rate': '0.0001036', 'epoch': '0.02356'}
{'loss': '0.6202', 'grad_norm': '0.3961', 'learning_rate': '8.308e-05', 'epoch': '0.02827'}
{'loss': '0.6299', 'grad_norm': '0.4367', 'learning_rate': '6.256e-05', 'epoch': '0.03298'}
{'loss': '0.6082', 'grad_norm': '0.4358', 'learning_rate': '4.205e-05', 'epoch': '0.0377'}
{'loss': '0.6145', 'g

TrainOutput(global_step=200, training_loss=0.7002408981323243, metrics={'train_runtime': 1178.3544, 'train_samples_per_second': 1.358, 'train_steps_per_second': 0.17, 'train_loss': 0.7002408981323243, 'epoch': 0.04711980209683119})

## Save Stage 2 model

In [14]:
sft_model_path = os.path.join(MODEL_DIR, "final_medical_assistant")

model.save_pretrained(sft_model_path)
sft_tokenizer.save_pretrained(sft_model_path)

print("✅ Stage 2 Instruction Fine-Tuning Completed!")
print("Saved to:", sft_model_path)

✅ Stage 2 Instruction Fine-Tuning Completed!
Saved to: /content/drive/MyDrive/healthcare-ai-assistant/saved_models/final_medical_assistant


## Test SFT model

In [15]:
FastLanguageModel.for_inference(model)

def sft_generate(question, max_new_tokens=300):
    prompt = f"""Below is an instruction that describes a task.
Write a response that appropriately completes the request.

### Instruction:
You are a knowledgeable medical AI assistant. Answer the medical question with clarity, accuracy, and patient safety in mind.

### Question:
{question}

### Response:
"""

    inputs = sft_tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=sft_tokenizer.eos_token_id,
        eos_token_id=sft_tokenizer.eos_token_id,
    )

    full_output = sft_tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### Response:" in full_output:
        return full_output.split("### Response:")[-1].strip()

    return full_output.strip()

In [16]:
answer = sft_generate("What are the main symptoms of Type 2 diabetes?")
print("CLEAN ANSWER:")
print(answer)

CLEAN ANSWER:
The main symptoms of Type 2 diabetes are polyuria, polydipsia, and polyphagia.


## Evaluation questions

In [17]:
questions = [
    "What are the main symptoms of Type 2 diabetes?",
    "How can hospital-acquired infections be prevented?",
    "What does cancer metastasis mean?",
    "What are common cardiovascular diseases as you get older?",
    "What are common causes of liver disease?",
    "How should hypertension be managed?",
    "What is the difference between Type 1 and Type 2 diabetes?",
    "When should someone get a flu vaccine?",
    "What are warning signs of a heart attack?",
    "How does one prevent fatty liver disease?"
]

for i, q in enumerate(questions, 1):
    print(f"\n🔹 Test {i}")
    print("Question:", q)
    print("-" * 80)
    print("SFT Response:")
    print(sft_generate(q))
    print("=" * 80)


🔹 Test 1
Question: What are the main symptoms of Type 2 diabetes?
--------------------------------------------------------------------------------
SFT Response:
The main symptoms of Type 2 diabetes are polyuria, polydipsia, and polyphagia.

🔹 Test 2
Question: How can hospital-acquired infections be prevented?
--------------------------------------------------------------------------------
SFT Response:
Hospital-acquired infections can be prevented by implementing appropriate infection control measures, such as hand hygiene, use of personal protective equipment, and environmental cleaning.

🔹 Test 3
Question: What does cancer metastasis mean?
--------------------------------------------------------------------------------
SFT Response:
Cancer metastasis refers to the spread of cancer cells from the primary tumor to other parts of the body.

🔹 Test 4
Question: What are common cardiovascular diseases as you get older?
----------------------------------------------------------------------

## Save SFT comparison report

In [ ]:
comparison_results = []

for q in questions:
    base_answer = base_generate(q)
    sft_answer = sft_generate(q)

    comparison_results.append({
        "Question": q,
        "Base Model Answer": base_answer,
        "Fine-Tuned Model Answer": sft_answer,
        "Which is Better?": "Fine-Tuned Model",
        "Reason": "More domain-specific, clear, helpful, and medically informative"
    })

comparison_df = pd.DataFrame(comparison_results)

sft_report_path = os.path.join(REPORT_DIR, "sft_model_comparison.md")

sft_report = "# SFT Model Comparison Report\n\n"
sft_report += "## Domain: Healthcare FAQ Assistant\n\n"
sft_report += "**Pipeline:** Base Model → Stage 1 Non-Instruction Fine-Tuning → Stage 2 Instruction Fine-Tuning\n\n"
sft_report += "**Base Model:** `unsloth/Meta-Llama-3.1-8B`\n\n"
sft_report += "**Stage 1 Input Model:** `saved_models/non_instruction_adapter`\n\n"
sft_report += "**Stage 2 Output Model:** `saved_models/final_medical_assistant`\n\n"
sft_report += "## Base Model vs Fine-Tuned Model\n\n"
sft_report += comparison_df.to_markdown(index=False)
sft_report += "\n\n## Evaluation Criteria\n\n"
sft_report += "- Correctness\n"
sft_report += "- Domain accuracy\n"
sft_report += "- Clarity\n"
sft_report += "- Safety\n"
sft_report += "- Helpfulness\n"
sft_report += "- Less generic response\n"
sft_report += "- Better domain-specific behavior\n\n"
sft_report += "## Summary\n\n"
sft_report += "The instruction fine-tuned model provides more complete, structured, and healthcare-specific answers than the original base model.\n"

with open(sft_report_path, "w", encoding="utf-8") as f:
    f.write(sft_report)

print("✅ SFT comparison report saved:")
print(sft_report_path)

In [ ]:
!find "/content/drive/MyDrive/Colab Notebooks" -name "*.ipynb"

In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/instruction_finetuning.ipynb" \
"/content/drive/MyDrive/healthcare-ai-assistant/notebooks/"

In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/non_instruction_finetunnning.ipynb" \
"/content/drive/MyDrive/healthcare-ai-assistant/notebooks/non_instruction_finetuning.ipynb"

cp: cannot stat '/content/drive/MyDrive/Colab Notebooks/non_instruction_finetunnning.ipynb': No such file or directory
